# Module 2: Model Evaluation & Interpretation

Choosing the right evaluation metric is as important as choosing the right model. A model that's 95% accurate sounds great — until you realize 95% of your data is one class and the model just predicts the majority class every time.

This notebook covers:
- Why accuracy is often misleading
- Precision vs recall tradeoffs
- ROC-AUC and when to use it
- SHAP for model interpretability

## 1. Setup: Creating an Imbalanced Classification Problem

We deliberately create an imbalanced dataset (70/30 split) because this is what real-world problems look like — fraud detection, churn, disease diagnosis. Balanced datasets are the exception, not the rule.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report,
)
import warnings
warnings.filterwarnings("ignore")

X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=5, weights=[0.7, 0.3], random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Class distribution: {np.bincount(y_train)}")
print(f"Minority class: {np.bincount(y_train)[1] / len(y_train):.1%}")

## 2. Training Multiple Models

**Why compare multiple models?** Different models have different strengths:
- **Logistic Regression** — fast, interpretable, good baseline
- **Random Forest** — handles non-linear relationships, robust to outliers
- **Gradient Boosting** — typically best performance on tabular data

We train all three to see how model complexity affects different metrics.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)

## 3. The Metrics That Matter

Here's the core insight most beginners miss:

| Metric | When to Use | Pitfall |
|--------|------------|--------|
| **Accuracy** | Balanced classes only | Misleading on imbalanced data |
| **Precision** | When false positives are costly (spam filter) | Ignores false negatives |
| **Recall** | When false negatives are costly (disease detection) | Ignores false positives |
| **F1** | When you need balance between precision and recall | Single number hides the tradeoff |
| **ROC-AUC** | Overall ranking quality, threshold-independent | Can be misleading with severe imbalance |
| **PR-AUC** | Imbalanced datasets (better than ROC-AUC) | Less intuitive to explain |

In [ ]:
header = f"{'Model':<25} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>6} {'PR-AUC':>7}"
print(header)
print("=" * 70)

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    print(f"{name:<25} {acc:>6.3f} {prec:>6.3f} {rec:>6.3f} {f1:>6.3f} {auc:>6.3f} {pr_auc:>7.3f}")

## 4. Confusion Matrix Deep Dive

The confusion matrix is the most honest view of model performance. Every other metric is just a different way of summarizing these four numbers:
- **True Positives (TP):** Correctly predicted positive
- **False Positives (FP):** Predicted positive, was negative (Type I error)
- **False Negatives (FN):** Predicted negative, was positive (Type II error)
- **True Negatives (TN):** Correctly predicted negative

In [ ]:
best_model = models["Gradient Boosting"]
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix (Gradient Boosting):")
print(f"                 Predicted Neg  Predicted Pos")
print(f"  Actual Neg     {cm[0][0]:>8}       {cm[0][1]:>8}")
print(f"  Actual Pos     {cm[1][0]:>8}       {cm[1][1]:>8}")
print(f"\nInterpretation:")
print(f"  {cm[0][1]} false positives - we flagged {cm[0][1]} negatives as positive")
print(f"  {cm[1][0]} false negatives - we MISSED {cm[1][0]} actual positives")
print(f"  In fraud detection, those {cm[1][0]} missed cases are real money lost.")

## 5. SHAP: Understanding WHY the Model Decides

**Why SHAP?** Metrics tell us HOW WELL the model performs, but SHAP tells us WHY it makes each prediction. This is critical for:
- **Debugging** — finding if the model learned spurious correlations
- **Trust** — explaining predictions to stakeholders
- **Compliance** — regulated industries require model explainability

SHAP values decompose each prediction into the contribution of each feature. A positive SHAP value pushes the prediction toward the positive class; negative pushes toward negative.

In [ ]:
import xgboost as xgb
import shap

xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    use_label_encoder=False, eval_metric="logloss", random_state=42,
)
xgb_model.fit(X_train, y_train)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

importance = np.abs(shap_values).mean(axis=0)
top_features = np.argsort(importance)[-10:][::-1]

print("Top 10 Features by Mean |SHAP| Value:")
print("-" * 40)
for rank, idx in enumerate(top_features, 1):
    print(f"  {rank:2d}. Feature {idx:2d}: {importance[idx]:.4f}")

print(f"\nBase rate (expected value): {explainer.expected_value:.4f}")

### Interpreting a Single Prediction

Let's decompose one prediction to see exactly how the model arrived at its decision. This is what you'd show a stakeholder who asks "why did the model flag this customer?" 

In [ ]:
sample_idx = 0
prediction = xgb_model.predict_proba(X_test[[sample_idx]])[:, 1][0]

print(f"Sample {sample_idx} - Predicted probability: {prediction:.4f}")
print(f"Base value: {explainer.expected_value:.4f}")
print(f"\nFeature contributions (top movers):")
print("-" * 45)

contributions = list(zip(range(len(shap_values[sample_idx])), shap_values[sample_idx]))
contributions.sort(key=lambda x: abs(x[1]), reverse=True)

for feat_idx, shap_val in contributions[:8]:
    direction = "INCREASES" if shap_val > 0 else "DECREASES"
    print(f"  Feature {feat_idx:2d}: {shap_val:+.4f} ({direction} risk)")

print(f"\nSum of all SHAP values + base = log-odds of prediction")

## Key Takeaways

1. **Accuracy is not enough** — on imbalanced data, always look at precision, recall, F1, and AUC
2. **Choose metrics based on business cost** — what's more expensive: a false positive or a false negative?
3. **ROC-AUC measures ranking quality** — "does the model rank positives higher than negatives?"
4. **SHAP provides per-prediction explanations** — essential for debugging, trust, and compliance
5. **The confusion matrix never lies** — always start there before looking at summary metrics